# ChaCha20



ChaCha is a **Stream Cipher**. That means, instead of scrabling the message directly, it first generates a long stream of random-looking bytes and then mixes the message with that stream.

As per the RFC-8439:

```txt
   ChaCha20 successively calls the ChaCha20 block function, with the
   same key and nonce, and with successively increasing block counter
   parameters.  ChaCha20 then serializes the resulting state by writing
   the numbers in little-endian order, creating a keystream block.
   Concatenating the keystream blocks from the successive blocks forms a
   keystream.  The ChaCha20 function then performs an XOR of this
   keystream with the plaintext.
```

It can be thought of as:

```bash
    Message:     HELLO
    Keystream:   Qx9@#
    Result:      (garbage)
```

The inputs to ChaCha20 are:
1. A 256-bit Key
2. A 96-bit nonce
3. A 32-bit initial counter
4. An arbitrary-length plaintext

The output is an encrypted message or "ciphertext", of the same length.

$\fbox{Note:}$ **Why aren't we allowed to reuse a number used once?**  
    &emsp;This is beacause if we reuse the same key paired with nonce, we get the same keystream thereby allowing attackers to deduce information. Since the message is XOR with the keystream.

### The Key

The key is the only secret in this algorithm. It determines:
- Who can generate the same keystream
- Who can decrypt the message

#### Properties:

- 256 bits
- Must remain secret
- Can be reused across many messages safely

The key is like the identifier of the encryption.

### The Nonce

The nonce ensures that the same key never reproduces the same keystream twice.

#### Properties:

- 96 bits
- Must be unique per key
- Can be public
- Random or sequential

The nonce is similar to the message ID.

### The Counter

ChaCha20 generates keystream in 64-byte chunks. The counter ensures that each chunk gets a different keystream, even within the same message.

## The Algorithm

### 1. Build the starting table

ChaCha20 starts by building a table of 16 numbers in 4x4 grid.

```bin
[ fixed words     ]   ← always the same
[ key words       ]   ← your secret
[ key words       ]
[ counter + nonce ]
```

|C|C|C|C|
|---|---|---|---|
|**K**|**K**|**K**|**K**|
|**K**|**K**|**K**|**K**|
|**B**|**N**|**N**|**N**|

Where, each cell is a 32 bit word

```bash
C - Constant
K - Key
B - 32-bit Block Counter
N - 96-bit Nonce split across 3 words
```

### 2. Copy the table

ChaCha20 immediately makes a copy of this table where one copy stays unchanged and the other copy gets aggressively scrambled.

### 3. Scramble the table

The algorithm now repeatedly scrambles the numbers in the table 20 times i.e 10 double rounds. Each double round = 1 column round and 1 diagonal round. It acieves this by adding numbers, flipping bits (XOR) and rotating bits in a circle.

#### The Mixing Operation

ChaCha20 repeatedly takes four numbers at a time and mixes them so that:
- Changing one bit affects many others
- Patterns disappear
- Output looks random
This operation is called a quarter round. Each quarter round operates on a fixed group of four words (either a column or diagonal).

#### Rounds

ChaCha20 does:
- Mix columns
- Then mix diagonals
- Then repeat

This happens 20 times. Why 20?
- Fewer rounds are faster but riskier
- 20 gives a huge safety margin
- No known attack breaks full 20 rounds

After these rounds, the table looks completely unrelated to the original inputs.

### 4. Add the original table back in

ChaCha20 now adds the scrambled table to the original table number by number. This feed-forward step prevents the internal permutation from being reversible.

### 5. Turn the table into bytes

The final table is now converted into bytes and is then produced as 64 bytes of output. These 64 bytes are the pure keystream.

### 6. Encrypt Message

ChaCha20 now combines the keystream and plaintext using XOR byte-by-byte.
$$
\text{Encrypted Data} = \text{Message} \oplus \text{Keystream}
$$

### 7. Move to Next Block

If the message is longer than 64 bytes:
- Increase the counter
- Build a new table
- Scramble again
- Produce the next keystream block
- Repeat